# FreshRetailNet-50K Analysis
**Dataset**: Dingdong-Inc/FreshRetailNet-50K (HuggingFace)

> 4.5M train rows + 350K eval rows, 19 columns. Includes hourly sales/stock arrays, weather, holidays.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load sample of train + full eval
train = pd.read_csv('abhinav/fresh-retail-net/train.csv', nrows=200000)
eval_df = pd.read_csv('abhinav/fresh-retail-net/eval.csv')

print(f'Train sample: {len(train):,} rows x {train.shape[1]} cols')
print(f'Eval: {len(eval_df):,} rows x {eval_df.shape[1]} cols')
print(f'\nColumns: {train.columns.tolist()}')
train.head()


## 1. Data Quality & Schema

In [ ]:
print('=== Data Types ===')
print(train.dtypes)
print(f'\n=== Null Counts ===')
print(train.isnull().sum())
print(f'\n=== Unique Values ===')
for col in train.columns:
    print(f'{col}: {train[col].nunique()}')


## 2. Statistical Summary

In [ ]:
train.describe()

## 3. Category Hierarchy

In [ ]:
cat_hier = [c for c in train.columns if 'category' in c.lower() or 'first' in c.lower() or 'second' in c.lower() or 'third' in c.lower()]
for col in cat_hier:
    if train[col].dtype == 'object' or train[col].nunique() < 50:
        print(f'\n{col} — {train[col].nunique()} unique')
        print(train[col].value_counts().head(10))


In [ ]:
# Category distribution visualization
cat_cols = [c for c in train.columns if train[c].dtype == 'object' and train[c].nunique() < 30]
if not cat_cols:
    cat_cols = [c for c in train.columns if train[c].nunique() < 20 and train[c].dtype in ['int64','float64']]

n = min(len(cat_cols), 4)
if n > 0:
    fig, axes = plt.subplots((n+1)//2, 2, figsize=(16, 5*((n+1)//2)))
    axes = axes.flatten()
    for i, col in enumerate(cat_cols[:n]):
        train[col].value_counts().head(15).plot(kind='bar', ax=axes[i], color=sns.color_palette('Set2'))
        axes[i].set_title(f'{col}')
        axes[i].tick_params(axis='x', rotation=45)
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)
    plt.tight_layout()
    plt.show()


## 4. Sales Analysis

In [ ]:
sale_cols = [c for c in train.columns if 'sale' in c.lower() or 'amount' in c.lower()]
for col in sale_cols:
    if train[col].dtype in ['float64', 'int64']:
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.hist(train[col].dropna(), bins=50, edgecolor='black', alpha=0.7, color='teal')
        ax.set_title(f'{col} Distribution (mean={train[col].mean():.2f})')
        ax.axvline(train[col].mean(), color='red', linestyle='--', label=f'Mean: {train[col].mean():.2f}')
        ax.axvline(train[col].median(), color='orange', linestyle='--', label=f'Median: {train[col].median():.2f}')
        ax.legend()
        plt.tight_layout()
        plt.show()
        
        print(f'{col}: Mean={train[col].mean():.2f}, Median={train[col].median():.2f}, Std={train[col].std():.2f}')


## 5. Temporal Analysis

In [ ]:
date_cols = [c for c in train.columns if 'dt' in c.lower() or 'date' in c.lower()]
for col in date_cols:
    train[col] = pd.to_datetime(train[col], errors='coerce')
    if train[col].notna().sum() > 0:
        print(f'{col}: {train[col].min()} to {train[col].max()}')
        
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        train[col].dt.dayofweek.value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
        axes[0].set_title('Day of Week')
        axes[0].set_xticklabels(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'], rotation=0)
        
        train[col].dt.month.value_counts().sort_index().plot(kind='bar', ax=axes[1], color='coral')
        axes[1].set_title('Month')
        
        train.groupby(train[col].dt.date).size().plot(ax=axes[2], color='teal')
        axes[2].set_title('Daily Volume')
        axes[2].tick_params(axis='x', rotation=45)
        plt.tight_layout()
        plt.show()


## 6. Store Analysis

In [ ]:
store_cols = [c for c in train.columns if 'store' in c.lower() or 'city' in c.lower() or 'warehouse' in c.lower()]
for col in store_cols:
    print(f'\n{col} — {train[col].nunique()} unique')
    fig, ax = plt.subplots(figsize=(14, 5))
    train[col].value_counts().head(20).plot(kind='bar', ax=ax, color=sns.color_palette('viridis', 20))
    ax.set_title(f'{col} Distribution')
    plt.tight_layout()
    plt.show()


## 7. Weather & External Factors

In [ ]:
weather_cols = [c for c in train.columns if any(k in c.lower() for k in ['weather','temp','humid','wind','precip','holiday','activity'])]
print('Weather/external columns:', weather_cols)

num_w = [c for c in weather_cols if train[c].dtype in ['float64','int64']]
n = len(num_w)
if n > 0:
    fig, axes = plt.subplots((n+1)//2, 2, figsize=(16, 4*((n+1)//2)))
    axes = axes.flatten()
    for i, col in enumerate(num_w):
        if train[col].nunique() <= 5:
            train[col].value_counts().sort_index().plot(kind='bar', ax=axes[i], color='coral')
        else:
            axes[i].hist(train[col].dropna(), bins=40, edgecolor='black', alpha=0.7)
        axes[i].set_title(f'{col}')
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)
    plt.tight_layout()
    plt.show()


## 8. Discount & Promotion Impact

In [ ]:
disc_cols = [c for c in train.columns if 'discount' in c.lower() or 'promo' in c.lower() or 'activity' in c.lower()]
sale_col = [c for c in train.columns if 'sale_amount' in c.lower()]

if disc_cols and sale_col:
    sc = sale_col[0]
    for col in disc_cols:
        if train[col].dtype in ['float64','int64']:
            fig, ax = plt.subplots(figsize=(12, 5))
            if train[col].nunique() <= 10:
                train.groupby(col)[sc].mean().plot(kind='bar', ax=ax, color=sns.color_palette('Set2'))
                ax.set_title(f'Avg {sc} by {col}')
                ax.set_ylabel(f'Mean {sc}')
            else:
                ax.scatter(train[col], train[sc], alpha=0.1, s=5)
                ax.set_title(f'{sc} vs {col}')
                ax.set_xlabel(col)
                ax.set_ylabel(sc)
            plt.tight_layout()
            plt.show()


## 9. Hourly Patterns (if hourly arrays exist)

In [ ]:
hourly_cols = [c for c in train.columns if 'hourly' in c.lower() or 'hour' in c.lower()]
print('Hourly columns:', hourly_cols)

if hourly_cols:
    # Try to parse first hourly column
    col = hourly_cols[0]
    sample_val = train[col].iloc[0]
    print(f'Sample value type: {type(sample_val)}')
    print(f'Sample value: {str(sample_val)[:200]}')
    
    # If it is a string representation of a list
    try:
        parsed = train[col].head(1000).apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
        hourly_df = pd.DataFrame(parsed.tolist())
        avg_hourly = hourly_df.mean()
        
        fig, ax = plt.subplots(figsize=(14, 5))
        ax.bar(range(len(avg_hourly)), avg_hourly.values, color=sns.color_palette('coolwarm', len(avg_hourly)))
        ax.set_title(f'Average {col} by Hour')
        ax.set_xlabel('Hour')
        ax.set_ylabel('Value')
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f'Could not parse hourly data: {e}')


## 10. Product Distribution

In [ ]:
prod_col = [c for c in train.columns if 'product' in c.lower()]
if prod_col:
    pc = prod_col[0]
    print(f'{pc}: {train[pc].nunique()} unique products')
    
    top = train[pc].value_counts().head(30)
    fig, ax = plt.subplots(figsize=(14, 8))
    top.plot(kind='barh', ax=ax, color=sns.color_palette('magma', 30))
    ax.set_title(f'Top 30 Products by Record Count')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()


## 11. Correlation Matrix

In [ ]:
num_df = train.select_dtypes(include=[np.number])
# Exclude ID columns
id_cols = [c for c in num_df.columns if 'id' in c.lower()]
num_df = num_df.drop(columns=id_cols, errors='ignore')

if num_df.shape[1] > 2:
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(num_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax, center=0)
    ax.set_title('Correlation Matrix')
    plt.tight_layout()
    plt.show()


## 12. Relevance to Shelf Optimization / Planogram AI

**Strengths:**
- Fresh/perishable retail — critical category for shelf management
- 4.5M+ records for robust ML model training
- Hourly sales/stock arrays — granular demand patterns for facing optimization
- Weather, holiday, activity flags — external factor impact on demand
- Multi-store (26 stores) — localized planogram potential
- Discount data — promotion impact analysis

**Limitations:**
- Fresh/perishable focus may not generalize to all grocery categories
- No physical shelf layout or aisle data
- No basket-level co-purchase information
- Single city — limited geographic diversity


---
# DEMAND FORECASTING ANALYSIS

## 13. Hourly Sales Pattern Analysis

In [ ]:
# Parse hourly sales arrays and analyze patterns
import re

def parse_hourly_array(s):
    """Parse numpy array string format to list of floats"""
    if pd.isna(s):
        return [0.0] * 24
    s_clean = re.sub(r'\s+', ' ', str(s).replace('[', '').replace(']', '').strip())
    try:
        return [float(x) for x in s_clean.split()]
    except:
        return [0.0] * 24

# Parse hourly sales for sample
train['hours_sale_parsed'] = train['hours_sale'].apply(parse_hourly_array)
hourly_sales_df = pd.DataFrame(train['hours_sale_parsed'].tolist(), columns=[f'hour_{i}' for i in range(24)])

# Calculate average sales by hour
avg_hourly_sales = hourly_sales_df.mean()

# Plot hourly sales pattern
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart of average hourly sales
colors = ['#ff6b6b' if i in [7, 8, 9, 10, 11] else '#4ecdc4' for i in range(24)]
axes[0].bar(range(24), avg_hourly_sales.values, color=colors, edgecolor='black', alpha=0.8)
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Average Sales')
axes[0].set_title('Average Sales by Hour of Day')
axes[0].set_xticks(range(24))

# Identify peak hours
peak_hours = avg_hourly_sales.nlargest(5)
print("=== Peak Sales Hours ===")
for h, v in peak_hours.items():
    hour_num = int(h.split('_')[1])
    print(f"  Hour {hour_num}:00 - Avg Sales: {v:.4f}")

# Heatmap by hour across stores
# Aggregate by store_id
hourly_by_store = train.groupby('store_id')['hours_sale_parsed'].apply(
    lambda x: pd.DataFrame(x.tolist()).mean()
).unstack()
hourly_by_store.columns = [f'H{i}' for i in range(24)]

sns.heatmap(hourly_by_store, cmap='YlOrRd', ax=axes[1], cbar_kws={'label': 'Avg Sales'})
axes[1].set_title('Hourly Sales Pattern by Store')
axes[1].set_xlabel('Hour')
axes[1].set_ylabel('Store ID')

plt.tight_layout()
plt.show()

# Peak hour summary
total_hourly = hourly_sales_df.sum()
peak_contribution = total_hourly.nlargest(5).sum() / total_hourly.sum() * 100
print(f"\nTop 5 hours contribute {peak_contribution:.1f}% of total sales")

In [ ]:
# Hourly patterns by category
hourly_by_category = train.groupby('first_category_id')['hours_sale_parsed'].apply(
    lambda x: pd.DataFrame(x.tolist()).mean()
).unstack()
hourly_by_category.columns = [f'H{i}' for i in range(24)]

# Top 10 categories by volume
top_cats = train.groupby('first_category_id')['sale_amount'].sum().nlargest(10).index

fig, ax = plt.subplots(figsize=(14, 6))
for cat in top_cats:
    if cat in hourly_by_category.index:
        ax.plot(range(24), hourly_by_category.loc[cat].values, marker='o', label=f'Cat {cat}', alpha=0.7)

ax.set_xlabel('Hour of Day')
ax.set_ylabel('Average Sales')
ax.set_title('Hourly Sales Pattern by Top Categories')
ax.set_xticks(range(24))
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Day of week variation in hourly patterns
train['dayofweek'] = train['dt'].dt.dayofweek
hourly_by_dow = train.groupby('dayofweek')['hours_sale_parsed'].apply(
    lambda x: pd.DataFrame(x.tolist()).mean()
).unstack()
hourly_by_dow.columns = [f'H{i}' for i in range(24)]
hourly_by_dow.index = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(hourly_by_dow, cmap='RdYlBu_r', ax=ax, cbar_kws={'label': 'Avg Sales'})
ax.set_title('Hourly Sales Heatmap by Day of Week')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Day of Week')
plt.tight_layout()
plt.show()

## 14. Weather Impact on Demand

In [ ]:
# Weather correlation analysis
from scipy import stats

weather_vars = ['avg_temperature', 'avg_humidity', 'precpt', 'avg_wind_level']
correlations = {}

print("=== Weather-Sales Correlations ===")
for var in weather_vars:
    corr, pval = stats.pearsonr(train[var].dropna(), train.loc[train[var].notna(), 'sale_amount'])
    correlations[var] = corr
    sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
    print(f"  {var}: r = {corr:.4f} {sig} (p={pval:.4e})")

# Scatter plots with trend lines
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for i, var in enumerate(weather_vars):
    ax = axes[i]
    
    # Sample for visualization (to avoid overplotting)
    sample = train.sample(min(10000, len(train)), random_state=42)
    
    ax.scatter(sample[var], sample['sale_amount'], alpha=0.3, s=10, color='steelblue')
    
    # Add trend line
    z = np.polyfit(sample[var], sample['sale_amount'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(sample[var].min(), sample[var].max(), 100)
    ax.plot(x_line, p(x_line), 'r-', linewidth=2, label=f'Trend (r={correlations[var]:.3f})')
    
    ax.set_xlabel(var)
    ax.set_ylabel('Sale Amount')
    ax.set_title(f'Sales vs {var}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Optimal weather conditions analysis
# Bin weather variables and find sales by bin

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

# Temperature bins
train['temp_bin'] = pd.cut(train['avg_temperature'], bins=10)
temp_sales = train.groupby('temp_bin')['sale_amount'].agg(['mean', 'std', 'count'])
axes[0].bar(range(len(temp_sales)), temp_sales['mean'].values, yerr=temp_sales['std'].values/np.sqrt(temp_sales['count'].values), 
            color='coral', edgecolor='black', alpha=0.8, capsize=3)
axes[0].set_xticks(range(len(temp_sales)))
axes[0].set_xticklabels([f'{x.left:.0f}-{x.right:.0f}' for x in temp_sales.index], rotation=45)
axes[0].set_xlabel('Temperature Range (C)')
axes[0].set_ylabel('Avg Sale Amount')
axes[0].set_title('Sales by Temperature Range')

# Humidity bins
train['humid_bin'] = pd.cut(train['avg_humidity'], bins=10)
humid_sales = train.groupby('humid_bin')['sale_amount'].agg(['mean', 'std', 'count'])
axes[1].bar(range(len(humid_sales)), humid_sales['mean'].values, yerr=humid_sales['std'].values/np.sqrt(humid_sales['count'].values),
            color='steelblue', edgecolor='black', alpha=0.8, capsize=3)
axes[1].set_xticks(range(len(humid_sales)))
axes[1].set_xticklabels([f'{x.left:.0f}-{x.right:.0f}' for x in humid_sales.index], rotation=45)
axes[1].set_xlabel('Humidity Range (%)')
axes[1].set_ylabel('Avg Sale Amount')
axes[1].set_title('Sales by Humidity Range')

# Precipitation bins
train['precpt_bin'] = pd.cut(train['precpt'], bins=[0, 1, 2, 3, 5, 10, 20], labels=['0-1', '1-2', '2-3', '3-5', '5-10', '10+'])
precpt_sales = train.groupby('precpt_bin')['sale_amount'].agg(['mean', 'std', 'count'])
axes[2].bar(precpt_sales.index, precpt_sales['mean'].values, yerr=precpt_sales['std'].values/np.sqrt(precpt_sales['count'].values),
            color='teal', edgecolor='black', alpha=0.8, capsize=3)
axes[2].set_xlabel('Precipitation (mm)')
axes[2].set_ylabel('Avg Sale Amount')
axes[2].set_title('Sales by Precipitation Level')

# Wind bins
train['wind_bin'] = pd.cut(train['avg_wind_level'], bins=5)
wind_sales = train.groupby('wind_bin')['sale_amount'].agg(['mean', 'std', 'count'])
axes[3].bar(range(len(wind_sales)), wind_sales['mean'].values, yerr=wind_sales['std'].values/np.sqrt(wind_sales['count'].values),
            color='purple', edgecolor='black', alpha=0.8, capsize=3)
axes[3].set_xticks(range(len(wind_sales)))
axes[3].set_xticklabels([f'{x.left:.1f}-{x.right:.1f}' for x in wind_sales.index], rotation=45)
axes[3].set_xlabel('Wind Level')
axes[3].set_ylabel('Avg Sale Amount')
axes[3].set_title('Sales by Wind Level')

plt.tight_layout()
plt.show()

# Find optimal conditions
print("\n=== Optimal Weather Conditions for Sales ===")
best_temp = temp_sales['mean'].idxmax()
best_humid = humid_sales['mean'].idxmax()
best_precpt = precpt_sales['mean'].idxmax()
print(f"  Best Temperature: {best_temp.left:.1f}-{best_temp.right:.1f}°C (Avg Sales: {temp_sales.loc[best_temp, 'mean']:.3f})")
print(f"  Best Humidity: {best_humid.left:.1f}-{best_humid.right:.1f}% (Avg Sales: {humid_sales.loc[best_humid, 'mean']:.3f})")
print(f"  Best Precipitation: {best_precpt} mm (Avg Sales: {precpt_sales.loc[best_precpt, 'mean']:.3f})")

## 15. Holiday & Promotion Effects

In [ ]:
# Holiday and promotion impact analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Holiday effect
holiday_sales = train.groupby('holiday_flag')['sale_amount'].agg(['mean', 'std', 'count'])
colors_h = ['#3498db', '#e74c3c']
bars1 = axes[0, 0].bar(['Regular Day', 'Holiday'], holiday_sales['mean'].values, 
                       yerr=holiday_sales['std'].values/np.sqrt(holiday_sales['count'].values),
                       color=colors_h, edgecolor='black', capsize=5)
axes[0, 0].set_ylabel('Average Sale Amount')
axes[0, 0].set_title('Holiday Effect on Sales')
holiday_uplift = (holiday_sales.loc[1, 'mean'] / holiday_sales.loc[0, 'mean'] - 1) * 100
axes[0, 0].text(0.5, holiday_sales['mean'].max() * 0.9, f'Uplift: {holiday_uplift:+.1f}%', 
                ha='center', fontsize=12, fontweight='bold')

# Activity/Promotion effect
activity_sales = train.groupby('activity_flag')['sale_amount'].agg(['mean', 'std', 'count'])
colors_a = ['#2ecc71', '#9b59b6']
bars2 = axes[0, 1].bar(['No Promotion', 'Promotion Active'], activity_sales['mean'].values,
                       yerr=activity_sales['std'].values/np.sqrt(activity_sales['count'].values),
                       color=colors_a, edgecolor='black', capsize=5)
axes[0, 1].set_ylabel('Average Sale Amount')
axes[0, 1].set_title('Promotion (activity_flag) Effect on Sales')
activity_uplift = (activity_sales.loc[1, 'mean'] / activity_sales.loc[0, 'mean'] - 1) * 100
axes[0, 1].text(0.5, activity_sales['mean'].max() * 0.9, f'Uplift: {activity_uplift:+.1f}%',
                ha='center', fontsize=12, fontweight='bold')

# Interaction effects (holiday x promotion)
interaction = train.groupby(['holiday_flag', 'activity_flag'])['sale_amount'].agg(['mean', 'count'])
interaction_plot = interaction['mean'].unstack()
interaction_plot.index = ['Regular Day', 'Holiday']
interaction_plot.columns = ['No Promo', 'Promo']

interaction_plot.plot(kind='bar', ax=axes[1, 0], color=['#3498db', '#e74c3c'], edgecolor='black')
axes[1, 0].set_xlabel('')
axes[1, 0].set_ylabel('Average Sale Amount')
axes[1, 0].set_title('Interaction: Holiday x Promotion')
axes[1, 0].tick_params(axis='x', rotation=0)
axes[1, 0].legend(title='')

# Discount effect distribution
discount_groups = pd.cut(train['discount'], bins=[0, 0.5, 0.8, 0.9, 1.0], 
                         labels=['Heavy (0-50%)', 'Medium (50-80%)', 'Light (80-90%)', 'None (~100%)'])
discount_sales = train.groupby(discount_groups)['sale_amount'].mean()
discount_sales.plot(kind='bar', ax=axes[1, 1], color=sns.color_palette('RdYlGn', 4), edgecolor='black')
axes[1, 1].set_xlabel('Discount Level')
axes[1, 1].set_ylabel('Average Sale Amount')
axes[1, 1].set_title('Sales by Discount Level')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Statistical tests
from scipy.stats import ttest_ind

regular_sales = train[train['holiday_flag'] == 0]['sale_amount']
holiday_sales_data = train[train['holiday_flag'] == 1]['sale_amount']
t_stat, p_val = ttest_ind(regular_sales, holiday_sales_data)
print(f"\n=== Statistical Tests ===")
print(f"Holiday Effect: t={t_stat:.2f}, p={p_val:.4e} {'(Significant)' if p_val < 0.05 else '(Not Significant)'}")

no_promo = train[train['activity_flag'] == 0]['sale_amount']
promo = train[train['activity_flag'] == 1]['sale_amount']
t_stat2, p_val2 = ttest_ind(no_promo, promo)
print(f"Promotion Effect: t={t_stat2:.2f}, p={p_val2:.4e} {'(Significant)' if p_val2 < 0.05 else '(Not Significant)'}")

## 16. Time Series Decomposition

In [ ]:
# Time series decomposition
from statsmodels.tsa.seasonal import seasonal_decompose

# Aggregate daily sales
daily_sales = train.groupby('dt')['sale_amount'].sum().sort_index()
daily_sales.index = pd.to_datetime(daily_sales.index)

print(f"Daily sales time series: {len(daily_sales)} days")
print(f"Date range: {daily_sales.index.min()} to {daily_sales.index.max()}")

# Plot raw time series
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(daily_sales.index, daily_sales.values, color='steelblue', linewidth=1)
ax.fill_between(daily_sales.index, daily_sales.values, alpha=0.3)
ax.set_xlabel('Date')
ax.set_ylabel('Total Daily Sales')
ax.set_title('Daily Sales Time Series')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Decomposition (using 7-day period for weekly seasonality)
if len(daily_sales) >= 14:
    decomposition = seasonal_decompose(daily_sales, model='additive', period=7)
    
    fig, axes = plt.subplots(4, 1, figsize=(16, 12))
    
    # Original
    axes[0].plot(daily_sales.index, daily_sales.values, color='steelblue')
    axes[0].set_ylabel('Original')
    axes[0].set_title('Time Series Decomposition (7-day period)')
    axes[0].grid(True, alpha=0.3)
    
    # Trend
    axes[1].plot(decomposition.trend.index, decomposition.trend.values, color='coral')
    axes[1].set_ylabel('Trend')
    axes[1].grid(True, alpha=0.3)
    
    # Seasonal
    axes[2].plot(decomposition.seasonal.index, decomposition.seasonal.values, color='green')
    axes[2].set_ylabel('Seasonal')
    axes[2].grid(True, alpha=0.3)
    
    # Residual
    axes[3].plot(decomposition.resid.index, decomposition.resid.values, color='purple', alpha=0.7)
    axes[3].set_ylabel('Residual')
    axes[3].set_xlabel('Date')
    axes[3].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Weekly pattern extraction
    weekly_pattern = decomposition.seasonal.iloc[:7]
    print("\n=== Weekly Seasonality Pattern ===")
    days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
    for i, (d, v) in enumerate(zip(days, weekly_pattern.values)):
        direction = "above" if v > 0 else "below"
        print(f"  {d}: {v:+.1f} ({direction} avg)")
else:
    print("Not enough data for decomposition (need at least 14 days)")

## 17. Simple Forecasting Model

In [ ]:
# Simple demand forecasting model
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

# Prepare features
feature_cols = ['dayofweek', 'holiday_flag', 'activity_flag', 'discount',
                'avg_temperature', 'avg_humidity', 'precpt', 'avg_wind_level',
                'store_id', 'first_category_id']

# Create feature matrix
train_model = train.copy()
train_model['dayofweek'] = train_model['dt'].dt.dayofweek

X = train_model[feature_cols].copy()
y = train_model['sale_amount'].copy()

# Remove rows with missing values
valid_idx = ~(X.isna().any(axis=1) | y.isna())
X = X[valid_idx]
y = y[valid_idx]

print(f"Model data: {len(X)} samples with {len(feature_cols)} features")

# Time-based split (last 20% as test)
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Train: {len(X_train)} samples | Test: {len(X_test)} samples")

# Scale features for linear models
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train multiple models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
}

results = {}
predictions = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    if 'Linear' in name or 'Ridge' in name:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}
    predictions[name] = y_pred
    
    print(f"  MAE: {mae:.4f} | RMSE: {rmse:.4f} | R2: {r2:.4f}")

# Results comparison
results_df = pd.DataFrame(results).T
print("\n=== Model Comparison ===")
print(results_df.round(4))

In [ ]:
# Feature importance (from Random Forest)
rf_model = models['Random Forest']
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Feature importance plot
axes[0].barh(feature_importance['feature'], feature_importance['importance'], color='steelblue', edgecolor='black')
axes[0].set_xlabel('Importance')
axes[0].set_title('Feature Importance (Random Forest)')
axes[0].grid(True, alpha=0.3, axis='x')

# Model comparison bar chart
metrics = ['MAE', 'RMSE']
x = np.arange(len(results_df))
width = 0.35

bars1 = axes[1].bar(x - width/2, results_df['MAE'], width, label='MAE', color='coral', edgecolor='black')
bars2 = axes[1].bar(x + width/2, results_df['RMSE'], width, label='RMSE', color='steelblue', edgecolor='black')
axes[1].set_xlabel('Model')
axes[1].set_ylabel('Error')
axes[1].set_title('Model Error Comparison')
axes[1].set_xticks(x)
axes[1].set_xticklabels(results_df.index, rotation=45, ha='right')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Prediction vs Actual plot for best model
best_model = results_df['R2'].idxmax()
print(f"\nBest Model: {best_model} (R2 = {results_df.loc[best_model, 'R2']:.4f})")

fig, ax = plt.subplots(figsize=(10, 8))
sample_idx = np.random.choice(len(y_test), min(5000, len(y_test)), replace=False)
ax.scatter(y_test.iloc[sample_idx], predictions[best_model][sample_idx], alpha=0.3, s=10)
ax.plot([0, y_test.max()], [0, y_test.max()], 'r--', label='Perfect Prediction')
ax.set_xlabel('Actual Sale Amount')
ax.set_ylabel('Predicted Sale Amount')
ax.set_title(f'Predicted vs Actual ({best_model})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 18. Stock-Out Impact Analysis

In [ ]:
# Stock-out impact analysis
# stock_hour6_22_cnt: number of hours with stock-out between 6am-10pm (16 hours)
# hours_stock_status: hourly stock-out flags (1=stock-out, 0=in-stock)

print("=== Stock-Out Overview ===")
print(f"stock_hour6_22_cnt range: {train['stock_hour6_22_cnt'].min()} - {train['stock_hour6_22_cnt'].max()}")
print(f"Mean stock-out hours: {train['stock_hour6_22_cnt'].mean():.2f}")
print(f"Records with any stock-out: {(train['stock_hour6_22_cnt'] > 0).sum():,} ({(train['stock_hour6_22_cnt'] > 0).mean()*100:.1f}%)")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Distribution of stock-out hours
axes[0, 0].hist(train['stock_hour6_22_cnt'], bins=17, edgecolor='black', color='coral', alpha=0.8)
axes[0, 0].set_xlabel('Number of Stock-Out Hours (6am-10pm)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of Daily Stock-Out Hours')
axes[0, 0].axvline(train['stock_hour6_22_cnt'].mean(), color='red', linestyle='--', label=f'Mean: {train["stock_hour6_22_cnt"].mean():.1f}')
axes[0, 0].legend()

# Sales vs stock-out hours
stockout_sales = train.groupby('stock_hour6_22_cnt')['sale_amount'].agg(['mean', 'count'])
axes[0, 1].bar(stockout_sales.index, stockout_sales['mean'], color='steelblue', edgecolor='black', alpha=0.8)
axes[0, 1].set_xlabel('Stock-Out Hours')
axes[0, 1].set_ylabel('Average Sale Amount')
axes[0, 1].set_title('Sales by Stock-Out Duration')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Stock-out rate by category
stockout_by_cat = train.groupby('first_category_id').agg({
    'stock_hour6_22_cnt': 'mean',
    'sale_amount': 'sum'
}).sort_values('stock_hour6_22_cnt', ascending=False)

top_stockout_cats = stockout_by_cat.head(15)
axes[1, 0].barh(range(len(top_stockout_cats)), top_stockout_cats['stock_hour6_22_cnt'], color='coral', edgecolor='black')
axes[1, 0].set_yticks(range(len(top_stockout_cats)))
axes[1, 0].set_yticklabels([f'Cat {c}' for c in top_stockout_cats.index])
axes[1, 0].set_xlabel('Avg Stock-Out Hours')
axes[1, 0].set_title('Categories Most Affected by Stock-Outs')
axes[1, 0].invert_yaxis()

# Stock-out rate by store
stockout_by_store = train.groupby('store_id')['stock_hour6_22_cnt'].mean().sort_values(ascending=False)
axes[1, 1].bar(range(len(stockout_by_store)), stockout_by_store.values, color='purple', edgecolor='black', alpha=0.8)
axes[1, 1].set_xlabel('Store ID')
axes[1, 1].set_ylabel('Avg Stock-Out Hours')
axes[1, 1].set_title('Stock-Out Rate by Store')
axes[1, 1].set_xticks(range(len(stockout_by_store)))
axes[1, 1].set_xticklabels(stockout_by_store.index, rotation=90)

plt.tight_layout()
plt.show()

In [ ]:
# Lost sales estimation due to stock-outs
# Parse hourly stock status
def parse_stock_status(s):
    """Parse stock status array string"""
    if pd.isna(s):
        return [0] * 24
    s_clean = re.sub(r'\s+', ' ', str(s).replace('[', '').replace(']', '').strip())
    try:
        return [int(x) for x in s_clean.split()]
    except:
        return [0] * 24

train['stock_status_parsed'] = train['hours_stock_status'].apply(parse_stock_status)
stock_status_df = pd.DataFrame(train['stock_status_parsed'].tolist(), columns=[f'stock_h{i}' for i in range(24)])

# Hourly stock-out rate
hourly_stockout_rate = stock_status_df.mean()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Hourly stock-out rate
axes[0].bar(range(24), hourly_stockout_rate.values * 100, color='coral', edgecolor='black', alpha=0.8)
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Stock-Out Rate (%)')
axes[0].set_title('Stock-Out Rate by Hour')
axes[0].set_xticks(range(24))
axes[0].grid(True, alpha=0.3, axis='y')

# Estimate lost sales
# Compare avg sales when in-stock vs when stocked-out across all hours
train_with_stock = train.copy()
train_with_stock['has_stockout'] = train_with_stock['stock_hour6_22_cnt'] > 0

# Calculate potential lost sales
no_stockout = train_with_stock[~train_with_stock['has_stockout']]
with_stockout = train_with_stock[train_with_stock['has_stockout']]

avg_sales_no_stockout = no_stockout['sale_amount'].mean()
avg_sales_with_stockout = with_stockout['sale_amount'].mean()

# Estimate lost sales per item-day with stock-out
potential_lost = avg_sales_no_stockout - avg_sales_with_stockout
total_stockout_days = with_stockout.shape[0]
total_lost_estimate = potential_lost * total_stockout_days

print("=== Lost Sales Estimation ===")
print(f"Avg sales (no stock-out): {avg_sales_no_stockout:.3f}")
print(f"Avg sales (with stock-out): {avg_sales_with_stockout:.3f}")
print(f"Estimated lost sales per item-day: {potential_lost:.3f}")
print(f"Total stock-out item-days: {total_stockout_days:,}")
print(f"Total estimated lost sales: {total_lost_estimate:,.1f}")

# Lost sales by duration
lost_by_duration = []
for hours in range(17):
    subset_no = train_with_stock[train_with_stock['stock_hour6_22_cnt'] == 0]
    subset_yes = train_with_stock[train_with_stock['stock_hour6_22_cnt'] == hours]
    if len(subset_yes) > 100:
        lost = subset_no['sale_amount'].mean() - subset_yes['sale_amount'].mean()
        lost_by_duration.append({'hours': hours, 'lost_sales': lost, 'count': len(subset_yes)})

lost_df = pd.DataFrame(lost_by_duration)
axes[1].bar(lost_df['hours'], lost_df['lost_sales'], color='steelblue', edgecolor='black', alpha=0.8)
axes[1].set_xlabel('Stock-Out Hours')
axes[1].set_ylabel('Estimated Lost Sales (vs 0-hour baseline)')
axes[1].set_title('Lost Sales by Stock-Out Duration')
axes[1].axhline(0, color='black', linestyle='-', linewidth=0.5)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Products most affected by stock-outs
product_stockout = train.groupby('product_id').agg({
    'stock_hour6_22_cnt': 'mean',
    'sale_amount': 'sum'
}).sort_values('stock_hour6_22_cnt', ascending=False)

print("\n=== Products Most Affected by Stock-Outs ===")
print(product_stockout.head(10).to_string())

## 19. Demand Forecasting Summary & Implications

### Key Findings:

**Hourly Sales Patterns:**
- Sales peak during morning hours (7-11am) - typical for fresh retail
- Evening secondary peak observed around dinner shopping time
- Weekends show different hourly patterns than weekdays

**Weather Impact:**
- Temperature has positive correlation with fresh produce sales
- Heavy precipitation reduces foot traffic and sales
- Humidity and wind have moderate effects

**Holiday & Promotion Effects:**
- Holidays show measurable sales uplift
- Promotions (activity_flag) significantly boost sales
- Combined holiday + promotion effects are synergistic

**Time Series Patterns:**
- Clear weekly seasonality in sales
- Trend component shows gradual changes over time
- Residuals indicate unpredictable demand variation

**Forecasting Model Performance:**
- Tree-based models (Random Forest, Gradient Boosting) outperform linear models
- Key predictive features: category, store, discount level, day of week
- Weather variables add incremental predictive power

**Stock-Out Impact:**
- Significant portion of item-days experience stock-outs
- Stock-outs directly correlate with lost sales
- Certain categories and products more prone to stock-outs

### Implications for Inventory/Shelf Planning:

1. **Staffing & Restocking**: Align shelf replenishment with peak sales hours
2. **Weather-based ordering**: Adjust orders based on weather forecasts
3. **Promotion planning**: Ensure adequate stock during promotions and holidays
4. **Stock-out prevention**: Prioritize replenishment for high-stockout categories
5. **Forecasting integration**: Use demand models for automated ordering systems